# Clinico-Transcriptomic Discovery of Prognostic Biomarkers and Functional Hallmarks in TCGA Breast Cancer Using Python
---
   ## Notebook 1: Data Acquisition, Integration, and Quality Control 


### 1. Motivation

Large-scale cancer transcriptomic studies require substantial preprocessing before meaningful biological analyses can be performed. Raw sequencing repositories frequently contain incomplete clinical annotations, duplicated identifiers, and inconsistencies between molecular and patient metadata.

Failure to address these issues can introduce bias into downstream analyses and compromise reproducibility.

This notebook establishes a high-quality analytical cohort by integrating RNA sequencing data with clinical information and applying a series of quality-control procedures.

Specifically, this notebook addresses the following questions:

> Can gene expression profiles be accurately linked to individual patients?

> Are there missing or inconsistent clinical records that should be excluded before analysis?

> Does the resulting dataset provide a reliable foundation for transcriptomic investigation?

The cleaned datasets generated here will serve as the input for all subsequent exploratory analyses and predictive modeling.

### 2. Import Required Libraries

In [1]:
import os
import requests
import json

import pandas as pd
import numpy as np

### 3. Load TCGA Gene Expression and Clinical Data

#### 3a. Obtain manifest file

In [2]:
files_endpt = "https://api.gdc.cancer.gov/files"

filters = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]}},
        {"op": "in", "content": {"field": "files.data_category", "value": ["Transcriptome Profiling"]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "files.analysis.workflow_type", "value": ["STAR - Counts"]}},
        {"op": "in", "content": {"field": "cases.samples.sample_type", "value": ["Primary Tumor"]}}
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields": "file_id,file_name,cases.submitter_id,cases.samples.submitter_id",
    "format": "JSON",
    "size": "1200"  
}

response = requests.get(files_endpt, params=params)
data = response.json()

file_list = []
for doc in data["data"]["hits"]:
    file_list.append({
        "file_id": doc["file_id"],
        "file_name": doc["file_name"],
        "patient_id": doc["cases"][0]["submitter_id"],
        "sample_id": doc["cases"][0]["samples"][0]["submitter_id"]
    })

manifest_df = pd.DataFrame(file_list)
print(f"Successfully mapped {manifest_df.shape[0]} primary tumor RNA-Seq files.")
manifest_df.to_csv("../data/manifest/gdc_manifest.csv", index=False)

Successfully mapped 1111 primary tumor RNA-Seq files.


In [3]:
manifest_for_client = manifest_df[['file_id']].rename(columns={'file_id': 'id'})
manifest_for_client.to_csv("../data/manifest/gdc_manifest.txt", sep="\t", index=False)
print("Manifest text file saved successfully!")

Manifest text file saved successfully!


#### 3b. Download TCGA_count files in command line

> Go to the GDC Data Transfer Tool website and download the version for your operating system (Windows, Mac, or Linux).
>
> Move the downloaded gdc-client file into your project repository.
>
> Open your terminal (or command prompt), navigate to your repository folder, and run this single command:
>
>```
>./gdc-client download -m data/manifest/gdc_manifest.txt -d data/raw/tcga_counts/
> ```

#### 3c. Download clinical data

In [4]:
cases_endpt = "https://api.gdc.cancer.gov/cases"

clinical_params = {
    "filters": json.dumps({
        "op": "in", "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]}
    }),
    "fields": (
        "submitter_id,"
        "demographic.vital_status,"
        "demographic.days_to_death,"
        "diagnoses.days_to_last_follow_up,"
        "diagnoses.treatments.treatment_type"
    ),
    "format": "JSON",
    "size": "1200"
}

print("Fetching expanded clinical metadata from GDC...")
response = requests.get(cases_endpt, params=clinical_params, timeout=30)
data = response.json()

clinical_records = []
for hit in data["data"]["hits"]:
    patient = hit["submitter_id"]
    
    demo = hit.get("demographic", {})
    vital_status = demo.get("vital_status", None)
    days_to_death = demo.get("days_to_death", None)
    
    diagnoses = hit.get("diagnoses", [])
    days_to_followup = None
    treatment_types = []
    
    if len(diagnoses) > 0:
        diagnosis = diagnoses[0]
        days_to_followup = diagnosis.get("days_to_last_follow_up", None)
        treatments = diagnosis.get("treatments", [])
        treatment_types = [t.get("treatment_type", "") for t in treatments if t.get("treatment_type")]
        
    clinical_records.append({
        "patient_id": patient,
        "vital_status": vital_status,
        "days_to_death": days_to_death,
        "days_to_last_follow_up": days_to_followup,
        "treatment_history": ",".join(treatment_types)
    })

clinical_df = pd.DataFrame(clinical_records)
print(clinical_df.head())

clinical_df.to_csv("../data/raw/tcga_clinical_raw.csv", index=False)
print("Updated clinical matrix saved successfully!")

Fetching expanded clinical metadata from GDC...
     patient_id vital_status  days_to_death  days_to_last_follow_up  \
0  TCGA-E9-A5FL        Alive            NaN                    24.0   
1  TCGA-A2-A1G4        Alive            NaN                   595.0   
2  TCGA-BH-A0HF        Alive            NaN                   727.0   
3  TCGA-AR-A1AS        Alive            NaN                  1150.0   
4  TCGA-D8-A13Y        Alive            NaN                  1728.0   

                                   treatment_history  
0  Surgery, NOS,Radiation Therapy, NOS,Pharmaceut...  
1  Chemotherapy,Radiation Therapy, NOS,Hormone Th...  
2  Hormone Therapy,Surgery, NOS,Radiation, Extern...  
3  Chemotherapy,Chemotherapy,Radiation, External ...  
4  Surgery, NOS,Radiation Therapy, NOS,Chemothera...  
Updated clinical matrix saved successfully!


#### 3d. Assemble Count files into count matrix

In [5]:
output_dir = "../data/raw/tcga_counts/"  

sample_frames = []

print("Assembling master raw counts matrix...")

for idx, row in manifest_df.iterrows():
    file_path = os.path.join(output_dir, row["file_id"], row["file_name"])
    
    if not os.path.exists(file_path):
        print(f"Warning: File missing, skipping -> {row['file_name']}")
        continue
        
    df = pd.read_csv(file_path, sep="\t", comment="#", skiprows=[1, 2, 3, 4])
    
    df.columns = [
        "gene_id", "gene_name", "gene_type", 
        "unstranded", "stranded_first", "stranded_second", 
        "tpm_unstranded", "fpkm_unstranded", "fpkm_uq_unstranded"
    ]
    
    counts_series = df.set_index("gene_id")["unstranded"]
    counts_series.name = row["sample_id"]
    sample_frames.append(counts_series)

tcga_raw_counts = pd.concat(sample_frames, axis=1)
print(f"Initial Raw Matrix Shape: {tcga_raw_counts.shape} (Includes pseudogenes, lncRNAs, etc.)")

print("\nFiltering for protein-coding genes...")

gene_metadata_map = df[["gene_id", "gene_type", "gene_name"]].copy()

protein_coding_genes = gene_metadata_map[gene_metadata_map["gene_type"] == "protein_coding"]
protein_coding_ids = protein_coding_genes["gene_id"].tolist()

tcga_raw_counts = tcga_raw_counts.loc[tcga_raw_counts.index.isin(protein_coding_ids)]

print("Assembly and Filtering Complete!")
print(f"Final Protein-Coding Matrix Shape: {tcga_raw_counts.shape} (Rows/Genes x Columns/Samples)")

os.makedirs("../data/raw/", exist_ok=True)
tcga_raw_counts.to_csv("../data/raw/tcga_brca_raw_counts.csv")
print("Filtered raw matrix saved successfully!")

Assembling master raw counts matrix...
Initial Raw Matrix Shape: (60660, 1111) (Includes pseudogenes, lncRNAs, etc.)

Filtering for protein-coding genes...
Assembly and Filtering Complete!
Final Protein-Coding Matrix Shape: (19962, 1111) (Rows/Genes x Columns/Samples)
Filtered raw matrix saved successfully!


### 4. Clean Clinical data

#### 4a. Clean survival data and remove empty patient records

In [6]:
clinical_df = pd.read_csv("../data/raw/tcga_clinical_raw.csv")
tcga_raw_counts = pd.read_csv("../data/raw/tcga_brca_raw_counts.csv")

print(f"Original clinical shape: {clinical_df.shape}")

# Create the binary event column (1 if Dead, 0 if Alive)
clinical_df["event"] = np.where(clinical_df["vital_status"] == 'Dead', 1, 0)

# Create a unified 'time' column based on vital status
clinical_df["time"] = np.where(
    clinical_df["vital_status"] == 'Dead', 
    clinical_df["days_to_death"], 
    clinical_df["days_to_last_follow_up"]
)

clinical_df = clinical_df.replace(r'^\s*$', np.nan, regex=True)
dead_and_blank_followup = (clinical_df["vital_status"] == "Dead") & (clinical_df["days_to_last_follow_up"].isna())
clinical_df.loc[dead_and_blank_followup, "days_to_last_follow_up"] = clinical_df.loc[dead_and_blank_followup, "days_to_death"]

clinical_df = clinical_df.dropna(subset=["vital_status", "days_to_death", "days_to_last_follow_up"], how="all")

clinical_df = clinical_df.dropna(subset=["time"])
clinical_df = clinical_df[clinical_df["time"] > 0]

print(f"Clinical shape after filtering anomalies: {clinical_df.shape}")

Original clinical shape: (1098, 5)
Clinical shape after filtering anomalies: (1021, 7)


#### 4b.  Clean treatment history (ONE-HOT ENCODING)

In [7]:
clinical_df['treatment_history'] = clinical_df['treatment_history'].fillna('')

clinical_df['had_surgery'] = clinical_df['treatment_history'].str.contains('Surgery', case=False).astype(int)
clinical_df['had_chemo'] = clinical_df['treatment_history'].str.contains('Pharmaceutical', case=False).astype(int)
clinical_df['had_radiation'] = clinical_df['treatment_history'].str.contains('Radiation', case=False).astype(int)

clinical_df = clinical_df.drop(columns=['treatment_history'])

### 5. Align Count Matrix with Clinical data

In [8]:
if "gene_id" in tcga_raw_counts.columns:
    tcga_raw_counts = tcga_raw_counts.set_index("gene_id")
else:
    tcga_raw_counts = tcga_raw_counts.copy()

sample_ids = tcga_raw_counts.columns.tolist()
patient_ids_from_counts = [sid[:12] for sid in sample_ids]

mapping_df = pd.DataFrame({
    "sample_id": sample_ids,
    "patient_id": patient_ids_from_counts
})

common_patients = set(mapping_df["patient_id"]).intersection(set(clinical_df["patient_id"]))
print(f"Number of common unique patients found: {len(common_patients)}")

clinical_cleaned = clinical_df[clinical_df["patient_id"].isin(common_patients)].copy()

clinical_cleaned = clinical_cleaned.merge(mapping_df, on="patient_id", how="inner")

clinical_cleaned = clinical_cleaned.sort_values("sample_id")

clinical_cleaned = clinical_cleaned.drop_duplicates(subset=["patient_id"], keep="first")
print(f"Clinical rows after strict deduplication: {clinical_cleaned.shape[0]}")

survival_sample_barcodes = clinical_cleaned["sample_id"].tolist()
counts_cleaned = tcga_raw_counts[survival_sample_barcodes].copy()

clinical_cleaned = clinical_cleaned.set_index("sample_id")
clinical_cleaned = clinical_cleaned.reindex(counts_cleaned.columns)

counts_cleaned.columns = [sid[:12] for sid in counts_cleaned.columns]

clinical_cleaned = clinical_cleaned.set_index("patient_id")

Number of common unique patients found: 1019
Clinical rows after strict deduplication: 1019


### 6. Quality assurance check

In [9]:
print("\n--- Final Matrix Verification ---")
print(f"Final Count Matrix Shape (Genes, Samples): {counts_cleaned.shape}")
print(f"Final Clinical Metadata Shape (Rows, Columns): {clinical_cleaned.shape}")

assert counts_cleaned.shape[1] == clinical_cleaned.shape[0], "CRITICAL ERROR: Matrix dimensions mismatch!"

perfect_identity = all(counts_cleaned.columns == clinical_cleaned.index)
print(f"Are columns and indices perfectly identical strings? {perfect_identity}")

counts_cleaned.to_csv("../data/processed/brca_counts_cleaned.csv")
clinical_cleaned.to_csv("../data/processed/brca_clinical_cleaned.csv")
print("\nPristine, aligned modeling files saved successfully!")


--- Final Matrix Verification ---
Final Count Matrix Shape (Genes, Samples): (19962, 1019)
Final Clinical Metadata Shape (Rows, Columns): (1019, 8)
Are columns and indices perfectly identical strings? True

Pristine, aligned modeling files saved successfully!


### Notebook Summary

In this notebook, raw RNA-seq sequencing files and clinical metadata were retrieved directly from the GDC repository and integrated into a unified analytical cohort. After restricting the analysis to protein-coding genes, constructing standardized survival variables, removing incomplete records, and resolving duplicate specimens, the final dataset consisted of 19,962 protein-coding genes measured across 1,019 uniquely matched primary breast tumor samples.

| Step                                   | Samples Remaining |
| -------------------------------------- | ----------------: |
| Downloaded RNA-seq files               |              1111 |
| Clinical records                       |              1098 |
| Valid survival information             |              1021 |
| Matched transcriptomic-clinical cohort |              1019 |


This curated dataset provides the foundation for all downstream analyses. The next notebook applies variance-stabilizing transformation and principal component analysis to characterize the global transcriptomic structure of the cohort before supervised statistical modeling.